In [ ]:
# Figure 5: tracking error histogram vs reward kernel exp(-e^2/std^2).
# If most errors sit left of the red line (max slope) the std can be reduced.
# Data: surviving robots, first second (50 steps) skipped so the start-up does not count.
# Every step of every robot is pooled into one histogram (all commands together).
lin_err = metrics.lin_vel_error_per_step(baseline_survivors)[:, 50:].ravel()
yaw_err = metrics.yaw_rate_error_per_step(baseline_survivors)[:, 50:].ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, err, term in zip(axes, [lin_err, yaw_err], ["track_linear_velocity", "track_angular_velocity"]):
    std = float(reward_cfg[term]["params"]["std"])
    e = np.linspace(0, 2 * std, 200)
    ax.hist(err, bins=80, range=(0, 2 * std), density=True, color="lightgray")
    ax.twinx().plot(e, np.exp(-e**2 / std**2))
    ax.axvline(std / np.sqrt(2), color="r")
    ax.set_title(f"{term} (std = {std})")
plt.show()

**[TO DELETE]** *Draft notes, move what we need into the [ANSWER] cell before submitting.*

**Figure 5.** If most errors sit well left of the red line, the kernel is flat there: shrink `std`, but keep the large errors where the kernel still gives reward (> ~0.1).

In [ ]:
# Figure 7: action smoothness and torque use.
# Action power above 3.5x the stride frequency is jitter; p99 torque per joint is compared to the actuator limits (red).
# Data: surviving robots, first second (50 steps) skipped for the spectrum and the torques,
# whole rollout for the jitter fraction (middle). p99 is taken over all steps and robots together.
cutoff = 3.5 * metrics.stride_frequency(r)
hf = metrics.action_hf_fraction(r, cutoff)
a = r.action[:, 50:] - r.action[:, 50:].mean(axis=1, keepdims=True)
freqs = np.fft.rfftfreq(a.shape[1], r.dt)
psd = (np.abs(np.fft.rfft(a, axis=1)) ** 2).mean(axis=(0, 2))
p99 = np.percentile(np.abs(r.joint_torque[:, 50:]), 99, axis=(0, 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].semilogy(freqs[1:], psd[1:])
axes[0].set(title="Action power spectrum", xlabel="Frequency (Hz)")
axes[1].boxplot([hf[r.cmd[:, 0] == v] for v in speeds], tick_labels=[f"{v:g}" for v in speeds])
axes[1].set(title="Power above 3.5x stride frequency", xlabel="vx command (m/s)")
axes[2].bar(range(12), p99)
axes[2].axhline(23.5, color="r")
axes[2].axhline(45, color="r")
axes[2].set(title="p99 |torque| per joint", xlabel="Joint index", ylabel="Torque (Nm)")
plt.show()

**[TO DELETE]** *Draft notes, move what we need into the [ANSWER] cell before submitting.*

**Figure 7.** Large power above 3.5x the stride frequency means jitter: raise `action_rate`. Torque near the red lines (23.5 Nm hip/thigh, 45 Nm calf) means raise `joint_torques`.

In [ ]:
# Figure 10: foot clearance per foot and commanded speed (no reward term sets the step height).
# Data: surviving robots, whole rollout; swings shorter than 2 steps are contact flickers and are skipped.
clear = np.full((r.num_robots, 4), np.nan)
for i in range(r.num_robots):
    for f in range(4):
        swings = metrics.phases(r.foot_contact[i, :, f], stance=False)
        peaks = [r.foot_height[i, s:e, f].max() for s, e in swings if e - s >= 2]
        if peaks:
            clear[i, f] = 100 * (np.mean(peaks) - metrics.FOOT_RADIUS)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for f, (ax, name) in enumerate(zip(axes, ["FL", "FR", "RL", "RR"])):
    data = [clear[r.cmd[:, 0] == v, f] for v in speeds]
    ax.boxplot([d[~np.isnan(d)] for d in data], tick_labels=[f"{v:g}" for v in speeds])
    ax.set(title=name, xlabel="vx command (m/s)")
axes[0].set_ylabel("Clearance (cm)")
plt.show()

**[TO DELETE]** *Draft notes, move what we need into the [ANSWER] cell before submitting.*

**Figure 10.** No reward term sets the step height. Compare the feet (diagonal asymmetry) and keep the low ones in mind for stairs.